# **LSTM Model Development and Evaluation Script**

---

## Modeling Pipeline

- **Baseline Model Development:** A standard model was trained using default/reasonable hyperparameters to establish a performance benchmark.

- **Hyperparameter Optimization:** Each baseline model was optimized using two complementary techniques:
  - ***`Manual Tuning`***: Efficient, adaptive search with pruning for rapid convergence to high-performing configurations.
  - ***`RandomizedSearchCV`***: Sklearn-compatible baseline optimization for fair comparison and reproducibility.

- **Cross-Validation and Robustness Assessment:** Each model variant was evaluated using ***`TimeSeriesSplit`*** to preserve temporal order and prevent look-ahead bias. Metrics were aggregated across folds to assess stability.

- **Overfitting Analysis:** A detailed comparison between cross-validation metrics and test set results was conducted. Additional metrics, including ***`RMSE ratio`*** and ***`R² gap`***, were computed to quantify overfitting and assess model generalization. ***`Directional accuracy`*** and ***`financial metrics`*** (Sharpe Ratio, Max Drawdown) were also calculated for trading-relevant evaluation.

---

## Persisted Artifacts

To ensure reproducibility, transparency, and extendability, the following artifacts have been saved for **each model**:

- **Optimized Model Performance:** Individual CSV files capturing the performance of each model variant:
    - ***LSTM (Baseline)***
    - ***LSTM (Manual Tuning)***
    - ***LSTM (RandomizedSearchCV)***

- **Best Variation Performance:** A CSV file containing only the metrics of the best-performing variation per model.

- **Summary of Model Performance:** A consolidated, extendable CSV file (`AllModel_OverallPerformance.csv`) including:
    - Cross-validation results (`CV MSE`, `CV MAE`, `CV RMSE`, `CV R²`, `CV MAPE`)
    - Test set results (`Test MSE`, `Test MAE`, `Test RMSE`, `Test R²`, `Test MAPE`)
    - Financial metrics (`Sharpe Ratio`, `Sortino Ratio`, `Max Drawdown`, `Directional Accuracy`)
    - Overfitting metrics (`R² gap`, `RMSE ratio`)
    - Overfitting status and model generalization label

- **Overfitting DataFrame:** An extendable CSV (`AllModel_OverfittingAnalysis.csv`) capturing overfitting analysis metrics across all models and variations.

- **Best Model per Algorithm:** The serialized best-performing variant of each algorithm for ensemble consideration or deployment.

- **Model Comparison:** A summary notebook or script that loads `AllModel_OverallPerformance.csv` and generates publication-ready comparison visualizations.

Together, these artifacts provide a complete, reproducible record of the modeling process, facilitating model tracking, comparison, selection, and deployment.

In [1]:
""" Configure the utilities module path for imports """
import sys
import os
from pathlib import Path

# get project root as parent of current working directory
PROJECT_ROOT = Path(os.getcwd()).parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
# artifacts root
DATA_ROOT = PROJECT_ROOT / "data"
FEATURE_ROOT = PROJECT_ROOT / "artifacts" / "FeatureSelection"
FIGURE_ROOT = PROJECT_ROOT / "visualizations" / "ModelEvaluation"
MODEL_ROOT = PROJECT_ROOT / "artifacts" / "Models"
PERFORMANCE_ROOT = PROJECT_ROOT / "artifacts" / "ModelPerformance"
MODEL_CHECKPOINT = MODEL_ROOT / "Checkpoints"

In [3]:
# records and calculations
import pandas as pd
import numpy as np

# vsiualizations
import matplotlib as plt
import seaborn as sns

# StandardScaler for scaling dataset
from sklearn.preprocessing import StandardScaler

# DL model setup
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Huber

import gc

# avoid minor warnings
import warnings
warnings.filterwarnings('ignore')
tf.get_logger().setLevel("ERROR")

# import utilities
from src.utilities import Evaluator, DataHandler, ExperimentTracker, ModelPersister

# Load Dataset and Artifact

In [4]:
df, x, y = DataHandler.load_dataset(DATA_ROOT / "gold_price_engineered.csv", target_col="target")
artifacts = DataHandler.load_artifacts(FEATURE_ROOT, cv_method='tscv')

In [5]:
# check dataset loading
df.head()

,Date,SPX_Return,GLD_Return,USO_Return,SLV_Return,EURUSD_Return,SPX_Return_lag1,SPX_Return_lag2,SPX_Return_lag3,SPX_Return_lag4,...,EURUSD_Return_lag5,GLD_Return_lag1,GLD_Return_lag2,GLD_Return_lag3,GLD_Return_lag4,GLD_Return_lag5,rolling_mean,rolling_std,momentum,target
0,2008-01-10,0.007948,0.019642,-0.016346,0.034858,0.009339,0.013624,-0.018352,0.003223,-0.024552,...,0.001902,-0.002650,0.023711,-0.004229,-0.005142,0.008367,0.006266,0.014169,0.011275,0.003739
1,2008-01-11,-0.013595,0.003739,-0.012564,0.000996,-0.000739,0.007948,0.013624,-0.018352,0.003223,...,0.000679,0.019642,-0.002650,0.023711,-0.004229,-0.005142,0.008043,0.012879,0.008881,0.010838
2,2008-01-14,0.010871,0.010838,0.015871,0.012627,0.005337,-0.013595,0.007948,0.013624,-0.018352,...,-0.004875,0.003739,0.019642,-0.002650,0.023711,-0.004229,0.011056,0.010901,0.015066,-0.017311
3,2008-01-15,-0.024925,-0.017311,-0.019798,-0.027396,-0.004499,0.010871,-0.013595,0.007948,0.013624,...,0.060478,0.010838,0.003739,0.019642,-0.002650,0.023711,0.002852,0.013993,-0.041022,-0.014661
4,2008-01-16,-0.005612,-0.014661,-0.012778,-0.011368,-0.009326,-0.024925,0.010871,-0.013595,0.007948,...,-0.058245,-0.017311,0.010838,0.003739,0.019642,-0.002650,0.000449,0.016053,-0.012010,-0.002307


In [6]:
# check input features
x.head()

,Date,SPX_Return,GLD_Return,USO_Return,SLV_Return,EURUSD_Return,SPX_Return_lag1,SPX_Return_lag2,SPX_Return_lag3,SPX_Return_lag4,...,EURUSD_Return_lag4,EURUSD_Return_lag5,GLD_Return_lag1,GLD_Return_lag2,GLD_Return_lag3,GLD_Return_lag4,GLD_Return_lag5,rolling_mean,rolling_std,momentum
0,2008-01-10,0.007948,0.019642,-0.016346,0.034858,0.009339,0.013624,-0.018352,0.003223,-0.024552,...,0.000679,0.001902,-0.002650,0.023711,-0.004229,-0.005142,0.008367,0.006266,0.014169,0.011275
1,2008-01-11,-0.013595,0.003739,-0.012564,0.000996,-0.000739,0.007948,0.013624,-0.018352,0.003223,...,-0.004875,0.000679,0.019642,-0.002650,0.023711,-0.004229,-0.005142,0.008043,0.012879,0.008881
2,2008-01-14,0.010871,0.010838,0.015871,0.012627,0.005337,-0.013595,0.007948,0.013624,-0.018352,...,0.060478,-0.004875,0.003739,0.019642,-0.002650,0.023711,-0.004229,0.011056,0.010901,0.015066
3,2008-01-15,-0.024925,-0.017311,-0.019798,-0.027396,-0.004499,0.010871,-0.013595,0.007948,0.013624,...,-0.058245,0.060478,0.010838,0.003739,0.019642,-0.002650,0.023711,0.002852,0.013993,-0.041022
4,2008-01-16,-0.005612,-0.014661,-0.012778,-0.011368,-0.009326,-0.024925,0.010871,-0.013595,0.007948,...,0.009339,-0.058245,-0.017311,0.010838,0.003739,0.019642,-0.002650,0.000449,0.016053,-0.012010


In [7]:
# check target feature
y.head()

0    0.003739
1    0.010838
2   -0.017311
3   -0.014661
4   -0.002307
Name: target, dtype: float64

In [8]:
# load train/test split data
x_train, x_test, y_train, y_test = artifacts['x_train'], artifacts['x_test'], artifacts['y_train'], artifacts['y_test']
cv = artifacts['cv']

# **Configuration Setup**

In [9]:
# random seeds for numpy and tensorflow
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [10]:
# core LSTM hyperparameters
SEQ_LEN = 20
LSTM_UNITS = [128, 64]
DROPOUT_RATE = 0.25
LEARNING_RATE = 1e-3

In [11]:
# training schedule and callback controls
EPOCHS = 200
BATCH_SIZE = 32
PATIENCE_ES = 25
PATIENCE_LR = 10
LR_FACTOR = 0.5
MIN_LR = 1e-6

In [12]:
# feature and target scaling
scaler_x = StandardScaler()
scaler_y = StandardScaler()

x_train_scaler = scaler_x.fit_transform(x_train)
x_test_scaler = scaler_x.transform(x_test)

y_train_scaler = scaler_y.fit_transform(y_train.values.reshape(-1, 1))
y_test_scaler = scaler_y.transform(y_test.values.reshape(-1, 1))

# **Build Baseline Model**

## Create Sequence and Define Callbacks

In [13]:
def create_sequences(x, y, seq_len):
    x_seq, y_seq = [], []
    
    for i in range(seq_len, len(x)):
        x_seq.append(x[i - seq_len:i])
        y_seq.append(y[i])

    return np.array(x_seq, dtype=np.float32), np.array(y_seq, dtype=np.float32)

In [14]:
x_train_seq, y_train_seq = create_sequences(x_train_scaler, y_train_scaler, SEQ_LEN)
x_test_seq, y_test_seq = create_sequences(x_test_scaler, y_test_scaler, SEQ_LEN)

In [15]:
print(f"x_train_seq shape : {x_train_seq.shape}")
print(f"x_test_seq  shape : {x_test_seq.shape}")
print(f"y_train_seq shape : {y_train_seq.shape}")
print(f"y_test_seq  shape : {y_test_seq.shape}")

x_train_seq shape : (1806, 20, 32)
x_test_seq  shape : (437, 20, 32)
y_train_seq shape : (1806, 1)
y_test_seq  shape : (437, 1)


In [16]:
# build lstm model
baseline_model = Sequential(name="LSTM_Baseline")

for idx, unit in enumerate(LSTM_UNITS):
    return_seq = idx < len(LSTM_UNITS) - 1

    baseline_model.add(LSTM(
        units=unit,
        return_sequences=return_seq,
        kernel_initializer="glorot_uniform",
        recurrent_initializer="orthogonal",
        name=f"lstm_{idx+1}",
    ))

    baseline_model.add(Dropout(
        DROPOUT_RATE,
        name=f"dropout_{idx+1}"
    ))

    baseline_model.add(BatchNormalization(
        name=f"batchnorm_{idx+1}"
    ))

baseline_model.add(Dense(1, activation="linear", name="output"))
loss = Huber(delta=1.0)

In [17]:
baseline_model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE, clipnorm=1.0),
    loss=loss,
    metrics=["mae"]
)

In [18]:
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE_ES,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=LR_FACTOR,
        patience=PATIENCE_LR,
        min_lr=MIN_LR,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=str(MODEL_CHECKPOINT / "LSTM_baseline_best.weights.h5"),
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=0
    )
]

## Train Baseline Model

In [19]:
history = baseline_model.fit(
    x_train_seq, y_train_seq,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=0,
    shuffle=False
)


Epoch 12: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 22: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
Epoch 28: early stopping
Restoring model weights from the end of the best epoch: 3.


In [20]:
baseline_model.summary()

Model: "LSTM_Baseline"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 20, 128)        │        82,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 20, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batchnorm_1                     │ (None, 20, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batchnorm_2                     │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 397,253 (1.52 MB)

 Trainable params: 132,289 (516.75 KB)

 Non-trainable params: 384 (1.50 KB)

 Optimizer params: 264,580 (1.01 MB)

In [21]:
# extract best validation performance
best_val_loss = min(history.history["val_loss"])
best_epoch = np.argmin(history.history["val_loss"])

print("Best Val Loss:", best_val_loss)
print("Best Epoch:", best_epoch)

Best Val Loss: 0.25497668981552124
Best Epoch: 2


## Apply Model to Make Prediction

In [22]:
# make prediction on train sequence
train_pred_scaled = baseline_model.predict(x_train_seq).flatten()
y_train_pred_baseline = scaler_y.inverse_transform(train_pred_scaled.reshape(-1, 1)).flatten()

# make prediction on test sequence
test_pred_scaled = baseline_model.predict(x_test_seq).flatten()
y_test_pred_baseline = scaler_y.inverse_transform(test_pred_scaled.reshape(-1, 1)).flatten()

57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step


## Evaluating the Model Performance

In [23]:
# evaluation of train metrics
train_metrics_baseline = Evaluator.calculate_metrics(y_train_seq, y_train_pred_baseline)

# evaluation of test metrics
test_metrics_baseline = Evaluator.calculate_metrics(y_test_seq, y_test_pred_baseline)

In [24]:
# calculate directional time-series specific accuracy
train_acc_baseline = Evaluator.directional_accuracy(y_train_seq, y_train_pred_baseline)
test_acc_baseline = Evaluator.directional_accuracy(y_test_seq, y_test_pred_baseline)

In [25]:
# calcluate financial metrics of test data for baseline model
fin_baseline = Evaluator.financial_metrics('LSTM (Baseline)', y_test_seq, y_test_pred_baseline)

display(fin_baseline)

,Model,Sharpe Ratio,Sortino Ratio,Max Drawdown,Total Return (%)
0,LSTM (Baseline),-0.0673,-0.1008,-100.0,-100.0


In [26]:
# baseline model performance
baseline_perf = Evaluator.performance_table(train_metrics_baseline + [train_acc_baseline], test_metrics_baseline + [test_acc_baseline])

print("LSTM - Baseline Modeling Performance")
display(baseline_perf)

LSTM - Baseline Modeling Performance


,Metrics,Training,Test
0,MSE,1.0016,0.3376
1,MAE,0.6853,0.4277
2,RMSE,1.0008,0.5810
3,R2 Score,0.0004,-0.0004
4,MAPE,100.2110,100.0769
5,Directional Accuracy (%),49.7927,50.0212


# **Structured Manual Tuning**

## Tuning Utility & Model Builder

In [27]:
# training utility
def run_experiment(
        build_model,
        x_train, y_train,
        x_test, y_test,
        scaler_y, experiment_name="ManualTuning",
        seq_len=SEQ_LEN, epochs=EPOCHS,
        batch_size=BATCH_SIZE, val_split=0.2,
        patience_es=PATIENCE_ES, patience_lr=PATIENCE_LR,
        lr_factor=LR_FACTOR, min_lr=MIN_LR,
        architecture=None, dropout=None,
        learning_rate=None, verbose=0
):
    
    # clear previous session to free memory
    tf.keras.backend.clear_session()
    gc.collect()

    # initialize model with input shape
    model = build_model(input_shape=(x_train.shape[1], x_train.shape[2]))

    # stops training when validation loss stops improving
    es = EarlyStopping(
        monitor="val_loss", patience=patience_es,
        restore_best_weights=True, verbose=0
    )

    # reduce learning rate when validation loss plateaus
    rlr = ReduceLROnPlateau(
        monitor="val_loss", factor=lr_factor,
        patience=patience_lr, min_lr=min_lr,  verbose=0
    )

    # saves best model weights based on validation loss
    mc = ModelCheckpoint(
        filepath=str(MODEL_CHECKPOINT / f"LSTM_{experiment_name}_best.weights.h5"),
        monitor="val_loss", save_best_only=True,
        save_weights_only=True, verbose=0
    )

    # train the model
    hist = model.fit(
        x_train, y_train,
        validation_split=val_split,
        epochs=epochs, batch_size=batch_size,
        callbacks=[es, rlr, mc],
        verbose=verbose, shuffle=False,
    )

    # make prediction on scaled input
    train_pred_scaled = model.predict(x_train).flatten()
    test_pred_scaled = model.predict(x_test).flatten()

    # inverse transform predictions back to original scale
    y_train_pred = scaler_y.inverse_transform(train_pred_scaled.reshape(-1, 1)).flatten()
    y_test_pred = scaler_y.inverse_transform(test_pred_scaled.reshape(-1, 1)).flatten()

    # compute regression metrics
    train_metrics = Evaluator.calculate_metrics(y_train, y_train_pred)
    test_metrics = Evaluator.calculate_metrics(y_test, y_test_pred)

    # compute directioal accuracy
    train_acc = Evaluator.directional_accuracy(y_train, y_train_pred)
    test_acc = Evaluator.directional_accuracy(y_test, y_test_pred)

    # extract best validation performance
    best_val_loss = min(hist.history["val_loss"])
    best_epoch = np.argmin(hist.history["val_loss"]) + 1

    # aggregate experiment result
    result = {
        "experiment": experiment_name,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "seq_len": seq_len,
        "architecture": architecture,
        "dropout": dropout,
        "learning_rate": learning_rate,
        "batch_size": batch_size,

        # train metrics
        "train_MSE": train_metrics[0],
        "train_MAE": train_metrics[1],
        "train_RMSE": train_metrics[2],
        "train_R2": train_metrics[3],
        "train_MAPE": train_metrics[4],
        "train_Dir_Acc": train_acc,

        # test metrics
        "test_MSE": test_metrics[0],
        "test_MAE": test_metrics[1],
        "test_RMSE": test_metrics[2],
        "test_R2": test_metrics[3],
        "test_MAPE": test_metrics[4],
        "test_Dir_Acc": test_acc,

    }

    tf.keras.backend.clear_session()
    gc.collect()
    
    return result, model, hist

In [28]:
# create LSTM model builder function for experiment
def lstm_builder(lstm_units, dropout=0.25, lr=1e-3, loss="huber", bidirectional=False):
    def builder(input_shape=(x_train.shape[1], x_train.shape[2])):
        model = Sequential(name=f"LSTM_{'_'.join(map(str, lstm_units))}")

        for idx, unit in enumerate(lstm_units):
            ret_seq = idx < len(lstm_units) - 1
            layer = LSTM(
                units=unit,
                return_sequences=ret_seq,
                kernel_initializer="glorot_uniform",
                recurrent_initializer="orthogonal",
                name=f"lstm_{idx+1}"
            )

            if bidirectional:
                layer = Bidirectional(layer, name=f"bilstm_{idx+1}")
            
            model.add(layer)
            model.add(Dropout(dropout, name=f"dropout_{idx+1}"))
            model.add(BatchNormalization(name=f"batchnorm_{idx+1}"))
        
        model.add(Dense(1, activation="linear", name="output"))
        loss = Huber(delta=1.0)

        model.compile(optimizer=Adam(learning_rate=lr, clipnorm=1.0),  loss=loss, metrics=["mae"])
        
        return model

    return builder

## Hyperparameter Tuning

### Sequence Length Tuning

In [29]:
SEQ_EXPERIMENTS = [10, 15, 20, 25, 30]

seq_results = []

for sl in SEQ_EXPERIMENTS:
    x_train, y_train = create_sequences(x_train_scaler, y_train_scaler, seq_len=sl)
    x_test, y_test = create_sequences(x_test_scaler, y_test_scaler, seq_len=sl)

    r, _, _ = run_experiment(
        lstm_builder([128, 64]),
        x_train, y_train,
        x_test, y_test,
        scaler_y=scaler_y,
        experiment_name=f"seq{sl}",
        seq_len=sl,
        epochs=200,
        batch_size=32,
        verbose=0,
    )
    seq_results.append(r)

best_seq = min(seq_results, key=lambda x: x["test_RMSE"])
BEST_SEQ_LEN = int(best_seq["seq_len"])
TEST_RMSE = round(best_seq["test_RMSE"], 4)

print(f">>> Best sequence length: {BEST_SEQ_LEN}  (test_RMSE={TEST_RMSE})")

57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
>>> Best sequence length: 30  (test_RMSE=0.5733)


In [30]:
# lock the best sequence for further tuning
x_train_seq, y_train_seq = create_sequences(x_train_scaler, y_train_scaler, BEST_SEQ_LEN)
x_test_seq, y_test_seq = create_sequences(x_test_scaler, y_test_scaler, BEST_SEQ_LEN)

### Architecture Tuning

In [31]:
ARCHITECTURES_EXPERIMENT = {
    "arch_1x64":         [64],
    "arch_1x128":        [128],
    "arch_2x64_32":      [64, 32],
    "arch_2x128_64":     [128, 64],
    "arch_3x128_64_32":  [128, 64, 32],
}

arch_result = []

for name, unit in ARCHITECTURES_EXPERIMENT.items():
    r, _, _ = run_experiment(
        lstm_builder(unit),
        x_train_seq, y_train_seq,
        x_test_seq, y_test_seq,
        scaler_y=scaler_y,
        experiment_name=name,
        seq_len=BEST_SEQ_LEN,
        architecture=unit,
        epochs=200,
        batch_size=32,
        verbose=0
    )

    arch_result.append(r)

best_arch = min(arch_result, key=lambda x: x["test_RMSE"])
BEST_UNIT = best_arch["architecture"]
TEST_RMSE = round(best_arch["test_RMSE"], 4)

print(f">>> Best architecture: {BEST_UNIT}  (test_RMSE={TEST_RMSE})")

57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
57/57 ━━━━━━━━━━━━━━━━━━━━ 3s 41ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
57/57 ━━━━━━━━━━━━━━━━━━━━ 3s 46ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
>>> Best architecture: [128]  (test_RMSE=0.5733)


### Dropout Tuning

In [32]:
DROPOUT_EXPERIMENT = [0.2, 0.25, 0.3, 0.4, 0.5]

drop_result = []

for drop in DROPOUT_EXPERIMENT:
    r, _, _ = run_experiment(
        lstm_builder(BEST_UNIT, dropout=drop),
        x_train_seq, y_train_seq,
        x_test_seq, y_test_seq,
        scaler_y=scaler_y,
        experiment_name=f"drop{int(drop*100)}",
        seq_len=BEST_SEQ_LEN,
        architecture=BEST_UNIT,
        dropout=drop,
        epochs=200,
        batch_size=32,
        verbose=0
    )

    drop_result.append(r)

best_drop = min(drop_result, key=lambda x: x["test_RMSE"])
BEST_DROPOUT = best_drop["dropout"]
TEST_RMSE = round(best_drop["test_RMSE"], 4)

print(f">>> Best dropout rate: {BEST_DROPOUT}  (test_RMSE={TEST_RMSE})")

57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
57/57 ━━━━━━━━━━━━━━━━━━━━ 3s 35ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
>>> Best dropout rate: 0.2  (test_RMSE=0.5733)


### Learning Rate Tuning

In [33]:
LR_EXPERIMENT =  [1e-4, 5e-4, 1e-3, 3e-3, 5e-3]

lr_result = []

for lr in LR_EXPERIMENT:
    r, _, _ = run_experiment(
        lstm_builder(BEST_UNIT, dropout=BEST_DROPOUT, lr=lr),
        x_train_seq, y_train_seq,
        x_test_seq, y_test_seq,
        scaler_y=scaler_y,
        experiment_name=f"lr{lr}",
        seq_len=BEST_SEQ_LEN,
        architecture=BEST_UNIT,
        dropout=BEST_DROPOUT,
        learning_rate=lr,
        epochs=200,
        batch_size=32,
        verbose=0
    )

    lr_result.append(r)

best_lr = min(lr_result, key=lambda x: x["test_RMSE"])

BEST_LR = best_lr["learning_rate"]
TEST_RMSE = round(best_lr["test_RMSE"], 4)

print(f">>> Best learning rate: {BEST_LR}  (test_RMSE={TEST_RMSE})")

57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
>>> Best learning rate: 0.003  (test_RMSE=0.5733)


### Batch Size Tuning

In [34]:
BATCH_EXPERIMENT = [16, 32, 64, 128]

batch_result = []

for batch in BATCH_EXPERIMENT:
    r, _, _ = run_experiment(
        lstm_builder(BEST_UNIT, dropout=BEST_DROPOUT, lr=BEST_LR),
        x_train_seq, y_train_seq,
        x_test_seq, y_test_seq,
        scaler_y=scaler_y,
        experiment_name=f"batch{batch}",
        seq_len=BEST_SEQ_LEN,
        architecture=BEST_UNIT,
        dropout=BEST_DROPOUT,
        learning_rate=BEST_LR,
        batch_size=batch,
    )

    batch_result.append(r)

best_batch = min(batch_result, key=lambda x: x["test_RMSE"])

BEST_BATCH = best_batch["batch_size"]
TEST_RMSE = round(best_batch["test_RMSE"], 4)

print(f">>> Best batch size: {BEST_BATCH}  (test_RMSE={TEST_RMSE})")

57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
57/57 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
>>> Best batch size: 64  (test_RMSE=0.5733)


### Loss Function Tuning

In [35]:
loss_result = []

In [36]:
r_mse, _, _ = run_experiment(
    lstm_builder(BEST_UNIT, dropout=BEST_DROPOUT, lr=BEST_LR, loss="mse"),
    x_train_seq, y_train_seq,
    x_test_seq, y_test_seq,
    scaler_y=scaler_y,
    experiment_name="loss-mse",
    seq_len=BEST_SEQ_LEN,
    epochs=200,
    batch_size=BEST_BATCH,
    verbose=0,
)

loss_result.append(r_mse)

57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step


In [37]:
r_huber, _, _ = run_experiment(
    lstm_builder(BEST_UNIT, dropout=BEST_DROPOUT, lr=BEST_LR, loss="huber"),
    x_train_seq, y_train_seq,
    x_test_seq, y_test_seq,
    scaler_y=scaler_y,
    experiment_name="loss-huber",
    seq_len=BEST_SEQ_LEN,
    epochs=200,
    batch_size=BEST_BATCH,
    verbose=0
)

loss_result.append(r_huber)

57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step


In [38]:
best_loss = min(loss_result, key=lambda x: x["test_RMSE"])

BEST_LOSS = best_loss["experiment"][:-4]                # remove 'loss' suffix
TEST_RMSE = round(best_loss["test_RMSE"], 4)

print(f">>> Best loss function: {BEST_LOSS}  (test_RMSE={TEST_RMSE})")

>>> Best loss function: loss  (test_RMSE=0.5734)


### Bidirectional Tuning

In [39]:
results_tracker = pd.DataFrame()

r, _, _ = run_experiment(
    lstm_builder(BEST_UNIT, dropout=BEST_DROPOUT, lr=BEST_LR, loss="huber", bidirectional=True),
    x_train_seq, y_train_seq,
    x_test_seq, y_test_seq,
    scaler_y=scaler_y,
    experiment_name="bidirectional",
    seq_len=BEST_SEQ_LEN,
    architecture=BEST_UNIT,
    dropout=BEST_DROPOUT,
    learning_rate=BEST_LR,
    batch_size=BEST_BATCH,
    epochs=200,
    verbose=0
)

row = pd.DataFrame([r])

results_tracker = pd.concat([results_tracker, row], ignore_index=True)

best_rmse = results_tracker[~results_tracker["experiment"].isin(["baseline",  "bidirectional"])]["test_RMSE"].min()

if r["test_RMSE"] < best_rmse:
    print(">>> Bidirectional is BETTER")
    USE_BIDIRECTIONAL = True

else:
    print(">>> Standard LSTM is better")
    USE_BIDIRECTIONAL = False

57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
>>> Standard LSTM is better


## Build and Train Manual Tuned Model

In [40]:
# configuration setup for final model training and evaluation
MANUAL_CONFIG = {
    "seq_len": BEST_SEQ_LEN,
    "lstm_units": BEST_UNIT,
    "dropout_rate": BEST_DROPOUT,
    "learning_rate": BEST_LR,
    "batch_size": BEST_BATCH,
    "loss_fn" : BEST_LOSS,
    "bidirectional": USE_BIDIRECTIONAL
    }

In [41]:
# create sequence for train and test set
x_train_seq, y_train_seq = create_sequences(x_train_scaler, y_train_scaler, MANUAL_CONFIG["seq_len"])
x_test_seq, y_test_seq = create_sequences(x_test_scaler, y_test_scaler, MANUAL_CONFIG["seq_len"])

In [42]:
tf.keras.backend.clear_session()
gc.collect()

0

In [43]:
# build final model with best configuration
manual_model = Sequential(name="LSTM_ManualTuning")

for idx, unit in enumerate(MANUAL_CONFIG["lstm_units"]):
    return_seq = idx < len(MANUAL_CONFIG["lstm_units"]) - 1
    
    layer = LSTM(
        units=unit,
        return_sequences=return_seq,
        kernel_initializer="glorot_uniform",
        recurrent_initializer="orthogonal",
        name=f"lstm_{idx+1}",
    )

    if MANUAL_CONFIG["bidirectional"]:
        layer = Bidirectional(layer, name=f"bilstm_{idx+1}")
    
    manual_model.add(layer)
    manual_model.add(Dropout(MANUAL_CONFIG["dropout_rate"], name=f"dropout_{idx+1}"))
    manual_model.add(BatchNormalization(name=f"batchnorm_{idx+1}"))

manual_model.add(Dense(1, activation="linear", name="output"))
loss_obj = Huber(delta=1.0) if MANUAL_CONFIG["loss_fn"] == "huber" else "mse"

In [44]:
# compile the model
manual_model.compile(
    optimizer=Adam(learning_rate=MANUAL_CONFIG["learning_rate"], clipnorm=1.0),
    loss=loss_obj,
    metrics=["mae"]
)

In [45]:
# model callbacks for final training
final_callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=30,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=15,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=str(MODEL_CHECKPOINT / "LSTM_ManualTuning_best.weights.h5"),
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=0
    )
]

In [46]:
#  fit the final model
history = manual_model.fit(
    x_train_seq, y_train_seq,
    validation_split=0.2,
    epochs=300,
    batch_size=MANUAL_CONFIG["batch_size"],
    callbacks=final_callbacks,
    verbose=0,
    shuffle=False
)


Epoch 17: ReduceLROnPlateau reducing learning rate to 0.001500000013038516.

Epoch 32: ReduceLROnPlateau reducing learning rate to 0.000750000006519258.
Epoch 32: early stopping
Restoring model weights from the end of the best epoch: 2.


In [47]:
# summary of the manual tuned model
manual_model.summary()

Model: "LSTM_ManualTuning"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 128)            │        82,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batchnorm_1                     │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 248,709 (971.52 KB)

 Trainable params: 82,817 (323.50 KB)

 Non-trainable params: 256 (1.00 KB)

 Optimizer params: 165,636 (647.02 KB)

In [48]:
# extract best validation performance
best_val_loss = round(min(history.history["val_loss"]), 4)
best_epoch = np.argmin(history.history["val_loss"]) + 1

print("Best Val Loss:", best_val_loss)
print("Best Epoch:", best_epoch)

Best Val Loss: 0.574
Best Epoch: 2


## Apply Model to Make Prediction

In [49]:
# make prediction on train sequence
train_pred_scaled = manual_model.predict(x_train_seq).flatten()
y_train_pred_manual = scaler_y.inverse_transform(train_pred_scaled.reshape(-1, 1)).flatten()

# make prediction on test sequence
test_pred_scaled = manual_model.predict(x_test_seq).flatten()
y_test_pred_manual = scaler_y.inverse_transform(test_pred_scaled.reshape(-1, 1)).flatten()

57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step


## Evaluation of Maual Tuned Model Performance

In [50]:
# evaluation of train metrics
train_metrics_manual = Evaluator.calculate_metrics(y_train_seq, y_train_pred_manual)

# evaluation of test metrics
test_metrics_manual = Evaluator.calculate_metrics(y_test_seq, y_test_pred_manual)

In [51]:
# calciulate directional accuracy
train_acc_manual = Evaluator.directional_accuracy(y_train_seq, y_train_pred_manual)
test_acc_manual = Evaluator.directional_accuracy(y_test_seq, y_test_pred_manual)

In [52]:
# calculate financial metrics of test data for manual tuned model
fin_manual = Evaluator.financial_metrics('LSTM (Manual Tuning)', y_test_seq, y_test_pred_manual)

display(fin_manual)

,Model,Sharpe Ratio,Sortino Ratio,Max Drawdown,Total Return (%)
0,LSTM (Manual Tuning),0.1704,0.2462,-100.0,-100.0


In [53]:
# manual tuning model performance
manual_perf = Evaluator.performance_table(train_metrics_manual + [train_acc_manual], test_metrics_manual + [test_acc_manual])

print("LSTM - Manual Tuning Performance")
display(manual_perf)

LSTM - Manual Tuning Performance


,Metrics,Training,Test
0,MSE,1.0016,0.3288
1,MAE,0.6853,0.4240
2,RMSE,1.0008,0.5735
3,R2 Score,0.0011,-0.0007
4,MAPE,100.1745,99.9218
5,Directional Accuracy (%),50.0130,49.5988
